# Cleaning 01: Profile and Deduplicate

Use this notebook to inspect the raw merge, remove fully duplicated rows, and enforce unique Inspection ID values before later cleaning stages.

In [4]:
import pandas as pd
from pathlib import Path

from logger import log_step, save_log

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

INTERIM_PATH = Path('../../data/interim/merged_inspections_licenses_inner.csv')
STAGE1_PATH = Path('../../data/interim/cleaning_stage1.csv')
STAGE1_LOG_PATH = Path('../../data/interim/cleaning_stage1_log.csv')

df_raw = pd.read_csv(INTERIM_PATH)
df_clean = df_raw.copy()

print('Loaded raw shape:', df_raw.shape)

log_step(
    step='Initial Load',
    rows_before=df_raw.shape[0],
    rows_after=df_clean.shape[0],
    cols_before=df_raw.shape[1],
    cols_after=df_clean.shape[1]
)

ImportError: cannot import name 'log_rows' from 'logger.logging' (d:\Science-the-Data\Food-Inspection\notebooks\logger\logging.py)

In [ ]:
def profile(df: pd.DataFrame, label: str) -> pd.DataFrame:
    missing = df.isna().sum().sort_values(ascending=False)
    missing_pct = (missing / len(df) * 100).round(2)

    out = pd.DataFrame({
        'missing_count': missing,
        'missing_pct': missing_pct,
        'dtype': df.dtypes.astype(str)
    }).sort_values(['missing_count', 'dtype'], ascending=[False, True])

    print(f'=== {label} ===')
    print('shape:', df.shape)
    print('full duplicates:', int(df.duplicated().sum()))
    print('duplicate Inspection ID:', int(df.duplicated(subset=['Inspection ID']).sum()))
    return out

pre_profile = profile(df_clean, 'Pre-clean profile')
pre_profile.head(20)

In [ ]:
# Explore 1: Identify fully-null columns
all_null_cols = [c for c in df_clean.columns if df_clean[c].isna().all()]
print('All-null columns:', all_null_cols)

In [ ]:
# Action 1: Drop fully-null columns
rows_before, cols_before = df_clean.shape
df_clean = df_clean.drop(columns=all_null_cols)
log_step('drop_all_null_columns', rows_before, df_clean.shape[0], cols_before, df_clean.shape[1], note='; '.join(all_null_cols))
print('Shape after dropping fully-null columns:', df_clean.shape)

In [ ]:
# Explore 2: Duplicate behavior
dup_rows = df_clean[df_clean.duplicated(subset=['Inspection ID'], keep=False)].sort_values('Inspection ID')
print('Rows in duplicate Inspection ID groups:', len(dup_rows))
print('Fully duplicated rows:', int(df_clean.duplicated().sum()))

if len(dup_rows) > 0:
    sample_id = dup_rows['Inspection ID'].iloc[0]
    sample_dup = dup_rows[dup_rows['Inspection ID'] == sample_id]
    print('Sample duplicate Inspection ID:', sample_id)
    print('Rows in sample group:', len(sample_dup))
    print('Sample rows identical?', sample_dup.nunique(dropna=False).eq(1).all())

In [ ]:
# Action 2: Remove duplicates (exact first, then by Inspection ID)
rows_before, cols_before = df_clean.shape

df_clean = df_clean.drop_duplicates(keep='first')
log_step('drop_exact_duplicates', rows_before, df_clean.shape[0], cols_before, df_clean.shape[1], note='Drop fully duplicated rows')

rows_before, cols_before = df_clean.shape

if 'Inspection Date' in df_clean.columns:
    sort_dates = pd.to_datetime(df_clean['Inspection Date'], errors='coerce')
    df_clean = df_clean.assign(_sort_inspection_date=sort_dates)
    df_clean = df_clean.sort_values(['Inspection ID', '_sort_inspection_date'], ascending=[True, False]).drop(columns=['_sort_inspection_date'])

df_clean = df_clean.drop_duplicates(subset=['Inspection ID'], keep='first')
log_step('drop_duplicate_inspection_id', rows_before, df_clean.shape[0], cols_before, df_clean.shape[1], note='Keep latest Inspection Date per Inspection ID')

expected_rows_after_dedup = 196_825
tolerance = int(expected_rows_after_dedup * 0.05)
actual_rows_after_dedup = df_clean.shape[0]
print('Shape after duplicate handling:', df_clean.shape)
print(f'Rows after dedup: {actual_rows_after_dedup:,} (expected ~{expected_rows_after_dedup:,} +/- {tolerance:,})')
assert abs(actual_rows_after_dedup - expected_rows_after_dedup) <= tolerance, 'Post-dedup row count is unexpectedly far from expected baseline.'
assert int(df_clean.duplicated(subset=['Inspection ID']).sum()) == 0, 'Duplicate Inspection ID values still remain after dedup.'

STAGE1_PATH.parent.mkdir(parents=True, exist_ok=True)
STAGE1_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(STAGE1_PATH, index=False)
pd.DataFrame(log_rows).to_csv(STAGE1_LOG_PATH, index=False)
print('Saved stage 1 data to:', STAGE1_PATH)
print('Saved stage 1 log to:', STAGE1_LOG_PATH)